In [1]:
import pandas as pd
from data import BasinDeltaTable

In [2]:
store = BasinDeltaTable(root_dir="./my_delta_dataset", overwrite=True)

In [3]:
# 2. Append data for multiple subbasins (this creates small files)
dates = pd.to_datetime(pd.date_range("2021-01-01", "2021-01-10", freq="D"))
data = pd.DataFrame(index=dates, data={'flow': range(10)})

for basin in range(10):
    data_dict = {}
    for subbasin in range(5):
        data_dict[f"{basin}_{subbasin}"] = data

    print(f"Upserting basin {basin}")
    store.upsert_dynamic(
        basin='mississippi',
        source='ERA5',
        data=data_dict
    )

Upserting 0
Table not found at my_delta_dataset/dynamic_data. Creating new Delta table.
Upserting 1
Upserting 2
Upserting 3
Upserting 4
Upserting 5
Upserting 6
Upserting 7
Upserting 8
Upserting 9


In [4]:
store.is_processed('mississippi','0_0','ERA5')

True

In [5]:
store.get_processing_status('mississippi','ERA5')

,basin,subbasin,source,processed_at,has_data
0,mississippi,9_0,ERA5,2025-09-18 15:28:09.975786+00:00,True
1,mississippi,9_1,ERA5,2025-09-18 15:28:09.975786+00:00,True
2,mississippi,9_2,ERA5,2025-09-18 15:28:09.975786+00:00,True
3,mississippi,9_3,ERA5,2025-09-18 15:28:09.975786+00:00,True
4,mississippi,9_4,ERA5,2025-09-18 15:28:09.975786+00:00,True
5,mississippi,8_0,ERA5,2025-09-18 15:28:09.887425+00:00,True
6,mississippi,8_1,ERA5,2025-09-18 15:28:09.887425+00:00,True
7,mississippi,8_2,ERA5,2025-09-18 15:28:09.887425+00:00,True
8,mississippi,8_3,ERA5,2025-09-18 15:28:09.887425+00:00,True
9,mississippi,8_4,ERA5,2025-09-18 15:28:09.887425+00:00,True


In [7]:
# 3. Read the data - it all works seamlessly
filtered_df = store.read_dynamic(basin='mississippi')
filtered_df

,flow,subbasin,year,basin,source
date,,,,,
2021-01-01 00:00:00+00:00,0,9_0,2021,mississippi,ERA5
2021-01-01 00:00:00+00:00,0,3_0,2021,mississippi,ERA5
2021-01-01 00:00:00+00:00,0,3_1,2021,mississippi,ERA5
2021-01-01 00:00:00+00:00,0,8_4,2021,mississippi,ERA5
2021-01-01 00:00:00+00:00,0,6_3,2021,mississippi,ERA5
...,...,...,...,...,...
2021-01-10 00:00:00+00:00,9,4_1,2021,mississippi,ERA5
2021-01-10 00:00:00+00:00,9,4_0,2021,mississippi,ERA5
2021-01-10 00:00:00+00:00,9,5_3,2021,mississippi,ERA5


In [8]:
store.compact()

Compacting processing metadata...
	Reduced file count by: 9 (files added: 10, files removed: 1).
Compacting dynamic_data...
	Reduced file count by: 9 (files added: 10, files removed: 1).
Vacuuming processing metadata...
Vacuuming dynamic data...


In [9]:
store.vacuum(retention_hours=0) 

Vacuuming processing metadata...
Vacuuming dynamic data...


In [10]:
store.table_uri

'my_delta_dataset/dynamic_data'